In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
#load data usign google file upload
from google.colab import files
uploaded = files.upload()

Saving RAW_recipes.csv to RAW_recipes.csv


In [7]:
import pandas as pd
data= pd.read_csv('RAW_recipes.csv')

In [8]:
data.head()

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


In [9]:
import pandas as pd
import ast
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense


In [10]:

# Drop rows with missing essential fields
df_clean = data.dropna(subset=["ingredients", "steps", "tags"])
df_clean = df_clean[df_clean["ingredients"].str.len() > 2]
df_clean = df_clean[df_clean["steps"].str.len() > 2]
df_clean = df_clean[df_clean["tags"].str.len() > 2]

# Convert stringified lists to actual Python lists
df_clean["ingredients"] = df_clean["ingredients"].apply(ast.literal_eval)
df_clean["steps"] = df_clean["steps"].apply(ast.literal_eval)
df_clean["tags"] = df_clean["tags"].apply(ast.literal_eval)

# Filter out empty lists
df_clean = df_clean[df_clean["ingredients"].apply(len) > 0]
df_clean = df_clean[df_clean["steps"].apply(len) > 0]
df_clean = df_clean[df_clean["tags"].apply(len) > 0]

# Convert to lowercase text strings
df_clean["ingredients_text"] = df_clean["ingredients"].apply(
    lambda x: ' '.join(i.lower() for i in x)
)
df_clean["tags_text"] = df_clean["tags"].apply(
    lambda x: ' '.join(t.lower().replace("-", " ") for t in x)
)
df_clean["steps_text"] = df_clean["steps"].apply(
    lambda x: ' '.join(s.lower() for s in x)
)
df_clean["steps_text"] = df_clean["steps_text"].apply(lambda x: "<start> " + x + " <end>")

# Final input string: ingredients + tags
df_clean["input_text"] = df_clean["ingredients_text"] + " " + df_clean["tags_text"]

# Select only the needed columns
final_df = df_clean[["input_text", "steps_text"]].reset_index(drop=True)

# Save to file (optional)
final_df.to_csv("preprocessed_recipes.csv", index=False)

print(final_df.head())


                                          input_text  \
0  winter squash mexican seasoning mixed spice ho...   
1  prepared pizza crust sausage patty eggs milk s...   
2  ground beef yellow onions diced tomatoes tomat...   
3  spreadable cheese with garlic and herbs new po...   
4  tomato juice apple cider vinegar sugar salt pe...   

                                          steps_text  
0  make a choice and proceed with recipe dependin...  
1  preheat oven to 425 degrees f press dough into...  
2  brown ground beef in large pot add chopped oni...  
3  place potatoes in a large pot of lightly salte...  
4  mix all ingredients& boil for 2 1 / 2 hours , ...  


In [19]:
# Load your preprocessed file
df = pd.read_csv("preprocessed_recipes.csv")

# Split the data
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

# Limit vocabulary sizes
max_input_vocab = 5000
max_target_vocab = 10000
max_target_len = 300  # Limit steps to 300 tokens

# Fit tokenizers on training data only
input_tokenizer = Tokenizer(num_words=max_input_vocab, oov_token="<OOV>")
input_tokenizer.fit_on_texts(train_df["input_text"])

target_tokenizer = Tokenizer(num_words=max_target_vocab, oov_token="<OOV>")
target_tokenizer.fit_on_texts(train_df["steps_text"])

# Convert texts to sequences
train_input_seq = input_tokenizer.texts_to_sequences(train_df["input_text"])
test_input_seq = input_tokenizer.texts_to_sequences(test_df["input_text"])

train_target_seq = target_tokenizer.texts_to_sequences(train_df["steps_text"])
test_target_seq = target_tokenizer.texts_to_sequences(test_df["steps_text"])

# Truncate long decoder sequences
train_target_seq = [seq[:max_target_len] for seq in train_target_seq]
test_target_seq = [seq[:max_target_len] for seq in test_target_seq]

# Get max lengths
max_input_len = max(len(seq) for seq in train_input_seq)
max_target_len = max(len(seq) for seq in train_target_seq)

# Pad sequences
train_encoder_input = pad_sequences(train_input_seq, maxlen=max_input_len, padding='post')
test_encoder_input = pad_sequences(test_input_seq, maxlen=max_input_len, padding='post')

train_target_padded = pad_sequences(train_target_seq, maxlen=max_target_len, padding='post')
test_target_padded = pad_sequences(test_target_seq, maxlen=max_target_len, padding='post')

# Prepare decoder inputs and outputs
train_decoder_input = train_target_padded[:, :-1]
train_decoder_output = train_target_padded[:, 1:]

test_decoder_input = test_target_padded[:, :-1]
test_decoder_output = test_target_padded[:, 1:]

# Expand decoder output for sparse_categorical_crossentropy
train_decoder_output = np.expand_dims(train_decoder_output, -1)
test_decoder_output = np.expand_dims(test_decoder_output, -1)

# Final vocab sizes
input_vocab_size = min(max_input_vocab, len(input_tokenizer.word_index) + 1)
target_vocab_size = min(max_target_vocab, len(target_tokenizer.word_index) + 1)

print("Input vocab size:", input_vocab_size)
print("Target vocab size:", target_vocab_size)
print("Train encoder input shape:", train_encoder_input.shape)
print("Train decoder input shape:", train_decoder_input.shape)
print("Train decoder output shape:", train_decoder_output.shape)


Input vocab size: 4412
Target vocab size: 10000
Train encoder input shape: (208472, 132)
Train decoder input shape: (208472, 299)
Train decoder output shape: (208472, 299, 1)


In [ ]:
import json

# Save word-to-index mapping for input (ingredients + tags)
with open("input_word2id.json", "w") as f:
    json.dump(input_tokenizer.word_index, f)

# Save word-to-index mapping for output (steps)
with open("target_word2id.json", "w") as f:
    json.dump(target_tokenizer.word_index, f)



In [28]:
import json

# Create reverse mapping: id → word
target_id2word = {str(v): k for k, v in target_tokenizer.word_index.items()}

# Save to JSON file
with open("target_id2word.json", "w") as f:
    json.dump(target_id2word, f)


In [20]:

embedding_dim = 256
lstm_units = 512
input_vocab_size = min(max_input_vocab, len(input_tokenizer.word_index) + 1)
target_vocab_size = min(max_target_vocab, len(target_tokenizer.word_index) + 1)

encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding = Embedding(input_vocab_size, embedding_dim)(encoder_inputs)
_, state_h, state_c = LSTM(lstm_units, return_state=True)(encoder_embedding)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_embedding = Embedding(target_vocab_size, embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(target_vocab_size, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])



In [21]:
#Model Training
history = model.fit(
    [train_encoder_input, train_decoder_input],
    train_decoder_output,
    batch_size=64,
    epochs=10,
    validation_split=0.1
)


Epoch 1/10
2932/2932 ━━━━━━━━━━━━━━━━━━━━ 157s 53ms/step - accuracy: 0.7422 - loss: 1.6123 - val_accuracy: 0.7953 - val_loss: 1.0358
Epoch 2/10
2932/2932 ━━━━━━━━━━━━━━━━━━━━ 158s 54ms/step - accuracy: 0.7990 - loss: 0.9997 - val_accuracy: 0.8056 - val_loss: 0.9379
Epoch 3/10
2932/2932 ━━━━━━━━━━━━━━━━━━━━ 158s 54ms/step - accuracy: 0.8081 - loss: 0.9133 - val_accuracy: 0.8104 - val_loss: 0.8948
Epoch 4/10
2932/2932 ━━━━━━━━━━━━━━━━━━━━ 153s 52ms/step - accuracy: 0.8130 - loss: 0.8690 - val_accuracy: 0.8130 - val_loss: 0.8712
Epoch 5/10
2932/2932 ━━━━━━━━━━━━━━━━━━━━ 153s 52ms/step - accuracy: 0.8158 - loss: 0.8425 - val_accuracy: 0.8149 - val_loss: 0.8566
Epoch 6/10
2932/2932 ━━━━━━━━━━━━━━━━━━━━ 153s 52ms/step - accuracy: 0.8178 - loss: 0.8249 - val_accuracy: 0.8160 - val_loss: 0.8471
Epoch 7/10
2932/2932 ━━━━━━━━━━━━━━━━━━━━ 153s 52ms/step - accuracy: 0.8197 - loss: 0.8092 - val_accuracy: 0.8171 - val_loss: 0.8391
Epoch 8/10
2932/2932 ━━━━━━━━━━━━━━━━━━━━ 154s 52ms/step - accuracy: 

In [14]:
model.save("lstm_recipe_model_tf.keras")

# Optional: Save training history for visualization
import pickle
with open("training_history", "wb") as f:
    pickle.dump(history.history, f)

print("Model training complete and saved.")


NameError: name 'history' is not defined

Model Evaluation

In [31]:
# ✅ LSTM Seq2Seq Inference on Test Data (Keras 3 Compatible)

import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pandas as pd

# ----------------------------
# 1. Load Tokenizers (JSON format)
# ----------------------------
with open("input_word2id.json") as f:
    input_word2id = json.load(f)

with open("target_word2id.json") as f:
    target_word2id = json.load(f)

with open("target_id2word.json") as f:
    target_id2word = json.load(f)

# Ensure special tokens exist
if "<start>" not in target_word2id:
    target_word2id["<start>"] = target_word2id.get("<OOV>", 1)

if "<end>" not in target_word2id:
    target_word2id["<end>"] = target_word2id.get("<OOV>", 1)

# ----------------------------
# 2. Load Test Data
# ----------------------------
test_df = pd.read_csv("preprocessed_recipes.csv").tail(1000)  # use last 1000 as test

def encode_input(text, word2id, maxlen):
    seq = [word2id.get(word, word2id['<OOV>']) for word in text.split()]
    return pad_sequences([seq], maxlen=maxlen, padding='post')

# ----------------------------
# 3. Match Vocab Sizes from Trained Model
# ----------------------------
embedding_dim = 256
lstm_units = 512
model = load_model("lstm_recipe_model_tf.keras")

# Dynamically match embedding weight shapes
encoder_vocab_size = model.get_layer("embedding_2").get_weights()[0].shape[0]
decoder_vocab_size = model.get_layer("embedding_3").get_weights()[0].shape[0]
max_input_len = 132

# ----------------------------
# 4. Rebuild Inference Models
# ----------------------------
# Encoder
encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding_layer = Embedding(encoder_vocab_size, embedding_dim)
encoder_embedding = encoder_embedding_layer(encoder_inputs)
encoder_lstm_layer = LSTM(lstm_units, return_state=True)
encoder_outputs, state_h_enc, state_c_enc = encoder_lstm_layer(encoder_embedding)
encoder_model = Model(encoder_inputs, [state_h_enc, state_c_enc])

# Decoder
decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_state_input_h = Input(shape=(lstm_units,), name="decoder_h")
decoder_state_input_c = Input(shape=(lstm_units,), name="decoder_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_layer = Embedding(decoder_vocab_size, embedding_dim)
decoder_embedding = decoder_embedding_layer(decoder_inputs)
decoder_lstm_layer = LSTM(lstm_units, return_sequences=True, return_state=True)
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm_layer(
    decoder_embedding, initial_state=decoder_states_inputs
)
decoder_dense_layer = Dense(decoder_vocab_size, activation="softmax")
decoder_outputs = decoder_dense_layer(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs, state_h_dec, state_c_dec]
)

# ----------------------------
# 5. Assign Weights from Trained Model
# ----------------------------
encoder_embedding_layer.set_weights(model.get_layer("embedding_2").get_weights())
encoder_lstm_layer.set_weights(model.get_layer("lstm_2").get_weights())

decoder_embedding_layer.set_weights(model.get_layer("embedding_3").get_weights())
decoder_lstm_layer.set_weights(model.get_layer("lstm_3").get_weights())
decoder_dense_layer.set_weights(model.get_layer("dense_1").get_weights())

# ----------------------------
# 6. Inference Function
# ----------------------------
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_word2id["<start>"]

    stop_condition = False
    decoded_sentence = []

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = target_id2word.get(str(sampled_token_index), "")

        if sampled_word == "<end>" or len(decoded_sentence) > 300:
            stop_condition = True
        else:
            decoded_sentence.append(sampled_word)

        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return " ".join(decoded_sentence)

# ----------------------------
# 7. Predict on Test Data
# ----------------------------
for i in range(5):
    input_text = test_df.iloc[i]['input_text']
    true_output = test_df.iloc[i]['steps_text']
    input_seq = encode_input(input_text, input_word2id, max_input_len)
    predicted_steps = decode_sequence(input_seq)

    print(f"\n[Example {i+1}]")
    print("Ingredients + Tags:", input_text[:150], "...")
    print("True Steps:", true_output[:200], "...")
    print("Predicted Steps:", predicted_steps[:200], "...")


/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━

Metric -BLEU

In [ ]:
#running on Test data and calculating metrics
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoothie = SmoothingFunction().method4
bleu_scores = []

for i in range(100):
    input_text = test_df.iloc[i]['input_text']
    true_output = test_df.iloc[i]['steps_text']
    input_seq = encode_input(input_text, input_word2id, max_input_len)
    pred_text = decode_sequence(input_seq)

    reference = [true_output.replace("<start>", "").replace("<end>", "").strip().split()]
    candidate = pred_text.strip().split()

    score = sentence_bleu(reference, candidate, smoothing_function=smoothie)
    bleu_scores.append(score)

    if i < 5:
        print(f"\n[Example {i+1}]")
        print("Ingredients + Tags:", input_text[:150], "...")
        print("True Steps:", true_output[:200], "...")
        print("Predicted Steps:", pred_text[:200], "...")

avg_bleu = round(sum(bleu_scores) / len(bleu_scores), 4)
print(f"\n✅ Average BLEU score on 100 test samples: {avg_bleu}")

NameError: name 'input_tokenizer' is not defined